In [13]:
import numpy as np
import pandas as pd
from caas_jupyter_tools import display_dataframe_to_user

def fitness_from_Sparams(freq,
                         S31_dB,
                         S21_dB,
                         deltaA_dB,
                         W=None,
                         Penalties=None,
                         bands=None,
                         lambda_reg=0.008,
                         eps=1e-12,
                         debug=False):
    """
    Calcula la función fitness siguiendo la sección 7.5 del trabajo.
    Entradas:
      - freq: array (Hz or GHz consistentemente; se usa tal cual para integraciones)
      - S31_dB, S21_dB, deltaA_dB: arrays con la misma longitud que freq (valores en dB)
      - W: matriz 5x3 de pesos (si None -> inicializa con 1s)
      - Penalties: matriz 5x3 de penalizaciones (si None -> ceros)
      - bands: lista de 3 tuplas con límites (fmin,fmax) en las mismas unidades que `freq`.
               Por defecto: [(65,70),(71,110),(111,115)] (valores mencionados en la tesis).
      - lambda_reg: término de regularización (por defecto 0.008 según la tesis).
      - eps: pequeño valor para estabilidad numérica en log.
      - debug: si True devuelve dict con matrices intermedias para inspección.
    Salida:
      - fitness (float)
      - si debug=True, devuelve (fitness, diagnostics_dict)
    """
    freq = np.asarray(freq)
    S31 = np.asarray(S31_dB)
    S21 = np.asarray(S21_dB)
    dA  = np.asarray(deltaA_dB)

    if freq.shape != S31.shape or freq.shape != S21.shape or freq.shape != dA.shape:
        raise ValueError("freq, S31_dB, S21_dB y deltaA_dB deben tener la misma forma.")

    # defaults
    if bands is None:
        bands = [(65,70), (71,110), (111,115)]

    # matrices W (5x3) y Penalties (5x3)
    if W is None:
        W = np.ones((5,3))
    if Penalties is None:
        Penalties = np.zeros((5,3))
    W = np.asarray(W, dtype=float)
    Penalties = np.asarray(Penalties, dtype=float)
    if W.shape != (5,3) or Penalties.shape != (5,3):
        raise ValueError("W y Penalties deben tener forma (5,3).")

    # helpers
    def indices_in_band(f, band):
        fmin, fmax = band
        return np.where((f >= fmin) & (f <= fmax))[0]

    def mse_to_target(y_region, target):
        if y_region.size == 0:
            return 0.0
        return float(np.mean((y_region - target)**2))

    def osc_intensity(y_region, f_region):
        """
        Oscillation intensity ~ integral of second derivative absolute value:
          Y = d^2/dx^2 (signal)
          Oscillation_intensity = |integral Y dx|
        Se calcula con gradient dos veces y trapz.
        """
        if y_region.size < 3:
            return 0.0
        dy = np.gradient(y_region, f_region)
        d2y = np.gradient(dy, f_region)
        val = np.abs(np.trapz(d2y, f_region))
        return float(val)

    # Construir F (5 x 3)
    F = np.zeros((5,3), dtype=float)

    for j, band in enumerate(bands):
        idx = indices_in_band(freq, band)
        f_region = freq[idx]
        # Valores en la región
        S31_r = S31[idx]
        S21_r = S21[idx]
        dA_r  = dA[idx]

        # 1) MSE S31 respecto a -3 dB (ecuación 7.22 idea)
        F[0,j] = mse_to_target(S31_r, -3.0)
        # 2) MSE S21
        F[1,j] = mse_to_target(S21_r, -3.0)
        # 3) MSE deltaA respecto a 0 dB
        F[2,j] = mse_to_target(dA_r, 0.0)
        # 4) Oscillation intensity S31 (ecuación 7.23-7.24)
        F[3,j] = osc_intensity(S31_r, f_region) if f_region.size>0 else 0.0
        # 5) Oscillation intensity S21
        F[4,j] = osc_intensity(S21_r, f_region) if f_region.size>0 else 0.0

    # Penalized weights
    Wpenalized = 100.0 * W + Penalties
    # Normalizar a [0,1]
    wmax = Wpenalized.max()
    wmin = Wpenalized.min()
    if np.isclose(wmax, wmin):
        Wnorm = np.zeros_like(Wpenalized)
    else:
        Wnorm = (Wpenalized - wmin) / (wmax - wmin)

    # Producto: (5x3) dot (3x5) => (5x5)
    product = Wnorm.dot(F.T)   # shape (5,5)

    # Log-scaling (evitar log(0) con eps)
    logscaled = np.log(100.0 * product + eps)

    # diagonal: vector de 5 elementos (cada uno representa un cost agregado por constraint)
    diag = np.diag(logscaled)   # length 5

    # fitness final: sum de los 3 primeros (smooth) + lambda * sum de los 2 últimos (oscillatory)
    fitness_value = float(np.sum(diag[0:3]) + lambda_reg * np.sum(diag[3:5]))

    if debug:
        diag_info = {
            "F": F,
            "W": W,
            "Penalties": Penalties,
            "Wpenalized": Wpenalized,
            "Wnorm": Wnorm,
            "product": product,
            "logscaled": logscaled,
            "diag": diag,
            "fitness": fitness_value,
            "bands": bands
        }
        return fitness_value, diag_info

    return fitness_value


ModuleNotFoundError: No module named 'pandas'

In [4]:
a = np.array([1,2,3,4,5,6])

b = np.where(a>3)

In [14]:
# --- Generar señales sintéticas ---
freq = np.linspace(65.0, 115.0, 401)  # en GHz, paso 0.125 GHz

# Base: fuera de bandas muy atenuado
S31 = np.full_like(freq, -60.0)
S21 = np.full_like(freq, -60.0)
deltaA = np.zeros_like(freq)

# Bandas: (65-70), (71-110), (111-115)
# Band 1: buen ajuste -3 dB con pequeña ondulación
idx1 = np.where((freq >= 65) & (freq <= 70))[0]
S31[idx1] = -3.0 + 0.4 * np.sin(8 * (freq[idx1] - 65))
S21[idx1] = -3.0 + 0.25 * np.sin(10 * (freq[idx1] - 65))
deltaA[idx1] = 0.0 + 0.2 * np.sin(6 * (freq[idx1] - 65))

# Band 2: algo peor, con overshoot/oscillaciones y un ligero desajuste
idx2 = np.where((freq >= 71) & (freq <= 110))[0]
S31[idx2] = -3.5 + 0.8 * np.sin(15 * (freq[idx2] - 71)) + 0.5 * np.exp(-(freq[idx2]-90)**2/200)
S21[idx2] = -2.8 + 0.6 * np.sin(12 * (freq[idx2] - 71)) - 0.4 * np.exp(-(freq[idx2]-95)**2/300)
deltaA[idx2] = 0.3 + 0.5 * np.sin(5 * (freq[idx2] - 71))  # desviación respecto 0

# Band 3: corto, con buen ajuste pero pequeña oscilación
idx3 = np.where((freq >= 111) & (freq <= 115))[0]
S31[idx3] = -3.0 + 0.3 * np.sin(9 * (freq[idx3] - 111))
S21[idx3] = -3.2 + 0.35 * np.sin(7 * (freq[idx3] - 111))
deltaA[idx3] = -0.05 + 0.15 * np.sin(4 * (freq[idx3] - 111))

# Matriz W proporcionada por el usuario
W = np.array([
    [0.005, 0.470, 0.001],
    [0.005, 0.470, 0.001],
    [0.001, 0.005, 0.001],
    [0.001, 0.005, 0.001],
    [0.001, 0.032, 0.001]
], dtype=float)

# Ejecutar fitness en modo debug para obtener matrices intermedias
fitness_val, diag_info = fitness_from_Sparams(freq, S31, S21, deltaA, W=W, Penalties=None, debug=True)

# Mostrar resultados
print(f"Fitness (valor): {fitness_val:.6f}")

# Mostrar la matriz F y otras matrices como dataframes para inspección
F_df = pd.DataFrame(diag_info["F"], index=[
    "MSE_S31_vs_-3dB",
    "MSE_S21_vs_-3dB",
    "MSE_deltaA_vs_0dB",
    "Osc_S31",
    "Osc_S21"
], columns=["Band1 (65-70)", "Band2 (71-110)", "Band3 (111-115)"])

W_df = pd.DataFrame(diag_info["W"], index=F_df.index, columns=F_df.columns)
Wpen_df = pd.DataFrame(diag_info["Wpenalized"], index=F_df.index, columns=F_df.columns)
Wnorm_df = pd.DataFrame(diag_info["Wnorm"], index=F_df.index, columns=F_df.columns)
product_df = pd.DataFrame(diag_info["product"], index=F_df.index, columns=F_df.index)
logscaled_df = pd.DataFrame(diag_info["logscaled"], index=F_df.index, columns=F_df.index)
diag_series = pd.Series(diag_info["diag"], index=F_df.index)

# Display tables to the user (interactive)
display_dataframe_to_user("Matriz F (valores de las métricas)", F_df)
display_dataframe_to_user("Matriz W (pesos proporcionados)", W_df)
display_dataframe_to_user("W penalizada (100*W + Penalties)", Wpen_df)
display_dataframe_to_user("W normalizada (0-1)", Wnorm_df)
display_dataframe_to_user("Producto Wnorm · F^T", product_df)
display_dataframe_to_user("Log-scaled (ln(100*product))", logscaled_df)
display_dataframe_to_user("Diagonal (valores usados en fitness)", diag_series.to_frame("diag_value"))

# También devolver arrays sintéticos (freq y deltaA) como un pequeño dataframe
flow_df = pd.DataFrame({
    "freq_GHz": freq,
    "S31_dB": S31,
    "S21_dB": S21,
    "deltaA_dB": deltaA
})
# show first & last rows for brevity
display_dataframe_to_user("Flujo sintético (primeras/últimas filas)", pd.concat([flow_df.head(8), flow_df.tail(8)]))

# Return the primary scalar for immediate visibility in the notebook output
fitness_val

Fitness (valor): 4.952781


/tmp/ipykernel_9413/4228723587.py:70: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  val = np.abs(np.trapz(d2y, f_region))


NameError: name 'pd' is not defined